# Sistema experto híbrido para el sector agrícola
## Riesgo de helada y estrés térmico en cultivos

**Unidad III** · Open-Meteo (API) + lógica difusa Mamdani + reglas SI-ENTONCES + Streamlit + IA generativa + mapa + notificaciones + trazabilidad.

**Flujo:** datos de la API → fuzzificación → grados de pertenencia → activación de reglas → agregación → defuzzificación → resultado → recomendación → explicación (IA) → notificación.

> **Principio de diseño:** la decisión la toma el motor experto (`experto/motor.py`). La IA generativa recibe el resultado ya calculado y solo lo explica.

**Estructura del proyecto:** `experto/` (módulos), `app.py` (interfaz Streamlit), este notebook y `salidas/` (trazas y alertas).

In [ ]:
# Dependencias (descomente si hace falta):
# %pip install -q requests pandas numpy matplotlib folium anthropic streamlit
import os, json
from datetime import date
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 110); pd.set_option("display.width", 220)

from experto import conocimiento as K, datos, motor, visual, ia, notificaciones, trazabilidad, pipeline
print("Huella de la base de conocimiento:", K.huella())

## 1. Exploración y selección de la fuente de datos

| Candidata | Contenido | Ventajas | Limitaciones | Decisión |
|---|---|---|---|---|
| **Open-Meteo** (pronóstico + histórico ERA5) | Temperatura, nubosidad, viento, humedad, punto de rocío, radiación, suelo, ET₀; horario; cualquier coordenada | Gratuita, sin clave, JSON simple, cobertura global, pasado y pronóstico, devuelve elevación | Rejilla de 10-25 km (suaviza microclimas) | **Seleccionada** |
| NASA POWER (agroclimatología) | Variables diarias agro-climáticas globales | Series largas | Resolución diaria (no ve la hora de la mínima ni la ventana nocturna), latencia | Descartada |
| Datos abiertos nacionales (p. ej. IDEAM en datos.gov.co) | Observaciones de estaciones / estadísticas agropecuarias | Mediciones reales | Cobertura puntual, variables y esquemas heterogéneos, huecos de datos | Descartada para el sistema (útil para validación) |

**Por qué Open-Meteo permite este sistema:** las heladas radiativas dependen de la *temperatura mínima*, del *cielo despejado* y del *aire en calma* durante la madrugada. Open-Meteo entrega exactamente esas tres variables **por hora**, lo que permite calcular la mínima del día y promediar viento y nubosidad en la ventana nocturna. Además, cada consulta es una coordenada, lo que da el componente geográfico de forma natural.

Variables relevantes: `temperature_2m` (°C), `cloud_cover` (%), `wind_speed_10m` (km/h). También se descargan `relative_humidity_2m` y `dew_point_2m` como contexto (no entran en las reglas).

In [ ]:
# Exploración de la estructura y calidad de la respuesta (Mendoza, 3 días de invierno)
r = datos.obtener_horario(-32.89, -68.84, "2026-07-10", "2026-07-12")
print("Endpoint:", r["endpoint"]); print("URL:", r["url"])
print("Elevación (m):", r["elevacion"], "| zona horaria:", r["zona_horaria"]); print("Unidades:", r["unidades"])
df = r["df"]; display(df.head(6)); print("Filas x columnas:", df.shape)
display(df.describe().T[["count", "min", "mean", "max"]])
print("Valores nulos por variable:\n", df.isna().sum())

## 2. Definición del problema

- **Situación:** riesgo de **helada** y de **estrés por calor** en un lote agrícola, para una ubicación y una fecha.
- **Información de entrada (4 variables):** temperatura mínima del día, temperatura máxima del día, viento medio nocturno y nubosidad media nocturna.
- **Salida del sistema experto:** índice de riesgo térmico (0-100) → nivel *bajo / moderado / alto / crítico* → tipo de riesgo (*helada* o *calor*) → recomendación.
- **Condición de notificación:** índice ≥ 60 (donde el nivel «alto» domina sobre «moderado»).

## 3. Consumo y preparación de los datos

Se consulta la API directamente y se transforman 24 valores horarios en 4 entradas. **Decisiones y justificación:**

| Entrada | Transformación | Justificación |
|---|---|---|
| `temp_min` | mínimo de las 24 h del día local | La helada depende del valor mínimo, no del promedio |
| `temp_max` | máximo de las 24 h | El estrés por calor depende del pico diurno |
| `viento` | promedio entre 00:00 y 07:59 | La helada radiativa se forma en la madrugada |
| `nubosidad` | promedio entre 00:00 y 07:59 | El cielo despejado nocturno favorece el enfriamiento |

Se usa `timezone=auto` para que «día» y «noche» sean locales, y se exige ≥ 20 horas válidas por día. Para demostrar el flujo con una condición fría real se busca (con **una** consulta al histórico) la noche más fría del invierno 2026 en Mendoza (Argentina).

In [ ]:
LAT, LON, NOMBRE = -32.89, -68.84, "Mendoza (Argentina)"
fecha_frio, resumen = datos.dia_extremo(LAT, LON, "2026-06-01", "2026-08-31", modo="frio")
print("Noche más fría del periodo:", fecha_frio, "->", resumen.loc[fecha_frio].to_dict())

consulta = datos.obtener_horario(LAT, LON, fecha_frio, fecha_frio)
prep = datos.preparar(consulta["df"], fecha_frio)
display(pd.DataFrame(prep["transformaciones"])[["variable", "origen", "operacion", "valor"]])

d = prep["horario_dia"]
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.plot(d["time"].dt.hour, d["temperature_2m"], marker="o", label="Temperatura 2 m (°C)")
ax.axvspan(K.VENTANA_NOCTURNA[0], K.VENTANA_NOCTURNA[1] + 1, color="navy", alpha=.12, label="Ventana nocturna")
ax.axhline(0, color="red", ls="--", lw=1, label="0 °C")
ax.set_xlabel("Hora local"); ax.set_title(f"{NOMBRE} · {fecha_frio}"); ax.legend(); ax.grid(alpha=.3); plt.show()

## 4. Variables y conjuntos difusos

Cuatro variables de entrada (se exigían ≥ 2) y una de salida. Los rangos que siguen un criterio técnico están indicados; el resto son **decisiones de modelado** declaradas como tales.

| Variable | Conjuntos | Referencia / justificación |
|---|---|---|
| Temperatura mínima | helada, fría, fresca, templada | Helada meteorológica = T ≤ 0 °C a 2 m (OMM). A nivel del cultivo la temperatura es 1-3 °C menor, por eso «helada» decrece hasta 2 °C *(modelado)* |
| Temperatura máxima | moderada, cálida, muy cálida | Estrés por calor desde ~30-35 °C en muchos cultivos; depende de la especie *(modelado, ajustable)* |
| Viento nocturno | calma, brisa, fuerte | La helada radiativa requiere aire en calma; cortes inspirados en Beaufort *(modelado)* |
| Nubosidad nocturna | despejado, parcial, nublado | Clasificación meteorológica en octas (≤2/8, 3-5/8, ≥6/8) con transiciones suaves |
| **Riesgo (salida)** | bajo, moderado, alto, crítico | Escala 0-100 con cuatro niveles traslapados *(modelado)* |

In [ ]:
fig = visual.graficas_todas(prep["entradas"]); plt.show()
for v, dfn in K.VARIABLES.items():
    print(f"\n{dfn['etiqueta']}\n  {dfn['justificacion']}")

## 5. Base de conocimiento (13 reglas SI-ENTONCES)

In [ ]:
reglas_df = pd.DataFrame([{"ID": x["id"], "Regla": K.texto_regla(x), "Tipo": x["tipo"]} for x in K.REGLAS])
display(reglas_df)

## 6. Motor de inferencia difuso (paso a paso)

Se implementó un motor **Mamdani propio** con NumPy: operador **Y = mínimo**, implicación por **recorte**, agregación por **máximo** y defuzzificación por **centroide**. Se muestra cada etapa con los datos de la API cargados arriba.

In [ ]:
# 6.1 Datos de la API -> entradas (ajustadas al universo de discurso)
usadas, recortes = motor.recortar_universo(prep["entradas"])
print("Entradas:", usadas, "| ajustes de universo:", recortes)

# 6.2 Fuzzificación -> grados de pertenencia
grados = motor.fuzzificar(usadas)
print("\nGrados de pertenencia:"); display(pd.DataFrame(grados).T.round(3))

# 6.3 Activación de reglas (fuerza = mínimo de los antecedentes)
reglas = motor.activar(grados)
act = pd.DataFrame([{"ID": x["id"], "Regla": x["texto"], "Fuerza": round(x["fuerza"], 3), "Limita": x["limitante"]}
                    for x in reglas if x["fuerza"] > 0])
print("Reglas activadas:"); display(act)

# 6.4 Agregación + 6.5 Defuzzificación (centroide) -> resultado
inf = motor.inferir(prep["entradas"])
visual.grafica_agregacion(inf); plt.show()
print(f"Resultado: {inf['resultado']} | nivel: {inf['nivel']} | tipo: {inf['tipo']}")
print("Pertenencia del resultado a cada nivel:", {k: round(v, 2) for k, v in inf["grados_salida"].items()})

## 7. Recomendación y explicabilidad

La recomendación sale de la **base de conocimiento** (tabla tipo × nivel), no de la IA. La explicabilidad se logra mostrando: datos usados → grados de pertenencia → reglas que dispararon (con la condición que limitó su fuerza) → índice → recomendación.

In [ ]:
r1 = pipeline.evaluar(LAT, LON, fecha_frio, nombre=NOMBRE, usar_ia=False, carpeta=None)   # demostración (no escribe archivos)
print("Recomendación:", r1["recomendacion"], "\n")
print("Explicación base (plantilla determinista):\n", r1["explicacion"]["texto"])

## 8. Interfaz visual

La interfaz está en `app.py` (Streamlit). Para ejecutarla, desde la carpeta del proyecto:

```bash
streamlit run app.py
```

Pestañas en el orden pedido: **Datos utilizados → Resultado → Reglas activadas → Recomendación → Explicación**, más **Mapa** y **Trazabilidad**. Inserte aquí capturas de pantalla como evidencia.

## 9. Integración de IA generativa

`experto/ia.py` arma el prompt **solo** con la salida del motor (variables, grados de pertenencia, reglas, resultado, recomendación) y le pide a Claude que lo **explique**. La instrucción de sistema prohíbe cambiar el nivel o la recomendación, y se verifica que el texto mencione el nivel calculado. Si no hay clave o falla la API, se usa la plantilla y el resultado no cambia.

In [ ]:
from getpass import getpass
API_KEY = os.getenv("ANTHROPIC_API_KEY") or ""
if not API_KEY:
    try: API_KEY = getpass("ANTHROPIC_API_KEY (Enter para omitir): ")
    except Exception: API_KEY = ""
USAR_IA = bool(API_KEY)
print("Prompt enviado a la IA:\n"); print(ia.construir_prompt(r1["contexto"]))
exp = ia.explicar(r1["contexto"], usar_ia=USAR_IA, api_key=API_KEY or None)
print("\nFuente:", exp["fuente"], "| modelo:", exp["modelo"], "| error:", exp["error"]); print(exp["texto"])

## 10. Componente geográfico

La ubicación se usa de forma funcional: (1) las coordenadas **determinan la consulta** a la API; (2) la API devuelve la **elevación**, que se registra en la traza y se pasa a la explicación; (3) el **mapa** colorea cada punto según el nivel de riesgo calculado por el motor.

In [ ]:
p = {**r1["ubicacion"], "riesgo": r1["inferencia"]["resultado"], "nivel": r1["inferencia"]["nivel"], "tipo": r1["inferencia"]["tipo"]}
visual.mapa_folium([p])

## 11. Notificaciones

**Condición:** índice de riesgo ≥ `K.UMBRAL_NOTIFICACION` (60), calculado por el motor experto. Se justifica porque desde ese valor domina el nivel «alto» y se requiere actuar antes de la noche. **Canales:** mensaje en la interfaz y archivo `salidas/alertas.jsonl` (+ `alertas.txt` legible). Se activa dentro del pipeline; se muestra su resultado en los casos de prueba de la sección 12.

## 12. Pruebas del sistema (4 casos con datos de la API)

Casos con condiciones contrastantes. Para los de helada y calor se selecciona con la API el día más extremo de un periodo; el caso 4 usa el **pronóstico de hoy**.

In [ ]:
CASOS = [
    dict(nombre="Caso 1 · Helada (Mendoza, AR)", lat=-32.89, lon=-68.84, buscar=("2026-06-01", "2026-08-31", "frio")),
    dict(nombre="Caso 2 · Calor (Valledupar, CO)", lat=10.46, lon=-73.25, buscar=("2026-07-01", "2026-09-30", "calor")),
    dict(nombre="Caso 3 · Noche fría de altiplano (Sabana de Bogotá, CO)", lat=4.86, lon=-74.03, buscar=("2026-07-01", "2026-09-30", "frio")),
    dict(nombre="Caso 4 · Condición normal hoy (Medellín, CO)", lat=6.25, lon=-75.57, buscar=None),
]
resultados = []
for c in CASOS:
    try:
        fecha = datos.dia_extremo(c["lat"], c["lon"], *c["buscar"])[0] if c["buscar"] else date.today()
        resultados.append(pipeline.evaluar(c["lat"], c["lon"], fecha, nombre=c["nombre"], usar_ia=USAR_IA, api_key=API_KEY or None))
    except Exception as exc:
        print("Falló", c["nombre"], "->", exc)

resumen = pd.DataFrame([{
    "Caso": x["ubicacion"]["nombre"], "Fecha": x["fecha"], **x["inferencia"]["entradas_usadas"],
    "Riesgo": x["inferencia"]["resultado"], "Nivel": x["inferencia"]["nivel"], "Tipo": x["inferencia"]["tipo"],
    "Reglas": ",".join(r["id"] for r in x["inferencia"]["reglas_activas"]),
    "Alerta": "SÍ" if x["notificacion"] else "no"} for x in resultados])
display(resumen)

In [ ]:
def mostrar_caso(x):
    inf = x["inferencia"]
    print("=" * 100); print(x["ubicacion"]["nombre"], "|", x["fecha"], "| elevación:", x["ubicacion"]["elevacion"], "m")
    print("\nDATOS USADOS:", inf["entradas_usadas"])
    print("\nGRADOS DE PERTENENCIA (> 0):")
    for v, cj in inf["fuzzificacion"].items():
        print(f"  {v}: ", {c: round(g, 2) for c, g in cj.items() if g > 0})
    print("\nREGLAS ACTIVADAS:")
    for r in inf["reglas_activas"]: print(f"  {r['id']} (fuerza {r['fuerza']:.2f}, limita: {r['limitante']}): {r['texto']}")
    print(f"\nRESULTADO: {inf['resultado']}/100 | nivel {inf['nivel']} | tipo {inf['tipo']}")
    print("RECOMENDACIÓN:", x["recomendacion"])
    print(f"EXPLICACIÓN ({x['explicacion']['fuente']}):", x["explicacion"]["texto"])
    n = x["notificacion"]; print("NOTIFICACIÓN:", n["titulo"] + " -> " + n["mensaje"] if n else "no se activó (riesgo < umbral)")
    visual.grafica_agregacion(inf); plt.show()

for x in resultados: mostrar_caso(x)

In [ ]:
# Mapa con los casos (color = nivel de riesgo del motor experto)
visual.mapa_folium([{**x["ubicacion"], "riesgo": x["inferencia"]["resultado"], "nivel": x["inferencia"]["nivel"],
                     "tipo": x["inferencia"]["tipo"]} for x in resultados])

In [ ]:
# Archivo de alertas generado por las notificaciones
alertas = notificaciones.leer_alertas("salidas")
display(pd.DataFrame(alertas)[["momento_utc", "titulo", "riesgo", "id_traza"]] if alertas else "Ningún caso superó el umbral.")

## 13. Trazabilidad

Cada ejecución guarda `salidas/trazas/traza_<id>.json` con: **dato de origen** (URL, fecha de consulta, SHA-256 de los datos y las 24 horas usadas) → **transformación** → **inferencia** → **reglas** → **resultado** → **recomendación** → **explicación/acción**. También se guarda la huella de la base de conocimiento, y `verificar()` reproduce la inferencia desde los datos guardados.

In [ ]:
ruta = resultados[0]["ruta_traza"]
traza = json.load(open(ruta, encoding="utf-8"))          # se reconstruye desde el archivo
print("Archivo:", ruta, "\n"); print(trazabilidad.cadena(traza))
print("\nVerificación de reproducibilidad:", trazabilidad.verificar(traza))

## 14. Conclusiones y limitaciones

- El sistema integra API real, lógica difusa, reglas, interfaz, IA generativa, mapa, notificaciones y trazabilidad, y **la decisión final siempre proviene del motor experto**.
- Los datos son de rejilla (~10-25 km): pueden subestimar heladas locales en valles; para uso operativo conviene contrastarlos con estaciones propias.
- Los rangos son generales y deben calibrarse con el cultivo y su etapa fenológica; los marcados como *modelado* son supuestos declarados.
- Mejoras posibles: pesos por regla, más variables (punto de rocío, humedad del suelo), series de varios días y validación contra heladas observadas.